<a href="https://colab.research.google.com/github/amonguscline/ML-Data-analysis/blob/main/Annoying_College.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from scipy.special import softmax

In [ ]:
lookuptable = pd.read_csv("/content/lookuptable.csv")
df = pd.read_csv("/content/college_data_scored.csv")

#Normalizing Data

In [ ]:
college_dupes={}
for i in range(len(df)):
  sender,subject=df.iloc[i,2],df.iloc[i,3]
  if sender in college_dupes:
    subdicts=college_dupes[sender]
    for subdict in subdicts:
      if subject in subdict:
        subdict[subject]+=1
        break
    else:
      subdicts.append({subject:1})
  else:
    college_dupes[sender]=[{subject:1}]

for sender,subdicts in college_dupes.items():
  total = 0
  for subdict in subdicts:
    dupe_n=list(subdict.values())[0]
    if dupe_n>1:
      total+=dupe_n
  college_dupes[sender]=total
print(college_dupes)

{'UCSC': 0, 'Oregon State': 9, 'Stanford': 0, 'UChicago': 40, 'Brandeis': 8, 'Grinnell': 26, 'Xavier': 4, 'Saint Louis University': 0, 'Colby': 14, 'Discovering U': 4, 'Miami University': 0, 'RPI': 0, 'CWRU': 8, 'Fordham': 2, 'Ulysses deArmas': 0, 'University of Denver': 4, 'George Mason University': 2, 'Northern Arizona University': 8, 'Syracuse University': 0, 'Baylor University': 2, 'Oberlin': 0, 'Stevens Institute of Technology': 0, 'Carleton College': 0, 'Yale': 2, 'Swarthmore': 2, 'JHU': 2, 'William & Mary': 2, 'BU': 2, 'University of Miami': 0, 'Arizona State': 6, 'HMC': 0, 'Occidental': 0, 'Caltech': 0, 'Biola': 18, 'VCU': 18, 'Northeastern': 16, 'NYU': 2, 'Chapman': 0, 'University of Notre Dame': 3, 'Williams College': 0, 'Harvard': 0, 'Lehigh': 27, 'Duke': 0, 'HSF': 22, 'Rose-Hulman': 56, 'University of San Francisco': 3, 'UMN': 2, 'Oklahoma University': 20, 'Naval Academy': 0, 'SMU': 4, 'Rutgers': 18, 'WPI': 16, 'MIT': 0, 'Bucknell': 4, 'Colgate': 6, 'Seton Hall': 94, 'India

In [ ]:
unique = pd.DataFrame(lookuptable["college name"].unique())
unique.columns = ["college name"]
unique["mean-ssl"],unique["total-emails"],unique["n-accounts"],unique["emoji-count"],unique["email-dupes"],unique["a-score"]=0,0,0,0,0,0
unique.reset_index(drop=True, inplace=True)
for i in range(len(unique["college name"])):
  unique.iloc[i,1]=int(df[df["sender"]==(unique.iloc[i,0])]["ssl"].mean())
  unique.iloc[i,2]=int(df[df["sender"]==(unique.iloc[i,0])]["sender"].count())
  unique.iloc[i,3]=int(lookuptable[lookuptable["college name"]==(unique.iloc[i,0])]["college name"].count())
  unique.iloc[i,4]=df[df["sender"]==(unique.iloc[i,0])]["emoji count"].sum()
  unique.iloc[i,5]=college_dupes[unique.iloc[i,0]]
  unique.iloc[i,6]=int(df[df["sender"]==(unique.iloc[i,0])]["annoyingness_score"].mean())

In [ ]:
#drop all rows with ssl=0
unique=unique[unique.iloc[:,1]!=0]
unique.reset_index(drop=True, inplace=True)

In [ ]:
columns=unique.columns[1:7]
unique_scaled=unique[columns].apply(lambda a:(a-a.min())/(a.max()-a.min()))
unique_softmax=unique_scaled[columns].apply(lambda a:softmax(a),axis=0)
final=pd.concat([unique.iloc[:,[0]],unique_softmax],axis=1)

In [ ]:
weights=[-2,2,2,1,3,1]
final["score"]=0
for i in range(len(final)):
  row=final.iloc[i]
  sum=0
  for j in range(1,7):
    sum+=row.iloc[j]*weights[j-1]
  final.iloc[i,7]=sum

<ipython-input-7-9ce0b03434a2>:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.05153212001107791' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final.iloc[i,7]=sum


#Scores

In [ ]:
top_10=final.sort_values(by="score",ascending=False).head(11)
top_10

,college name,mean-ssl,total-emails,n-accounts,emoji-count,email-dupes,a-score,score
52,Seton Hall,0.005197,0.009766,0.005866,0.005915,0.014823,0.006891,0.078146
100,UC Merced,0.005219,0.008149,0.006556,0.012308,0.011361,0.006703,0.072067
41,Rose-Hulman,0.005128,0.013552,0.005250,0.005915,0.009894,0.004065,0.067007
109,Texas A&M,0.005445,0.006401,0.012769,0.006239,0.008083,0.006340,0.064281
95,Columbia Pre College,0.005268,0.006346,0.014270,0.005607,0.006262,0.005077,0.060166
54,Willamette,0.005185,0.009724,0.006556,0.007268,0.007039,0.005220,0.055796
81,Puget Sound,0.005241,0.007508,0.005866,0.015241,0.005813,0.006519,0.055465
43,Oklahoma University,0.005228,0.007941,0.008187,0.007268,0.006746,0.005674,0.054980
10,CWRU,0.005169,0.011114,0.006556,0.005693,0.005937,0.005518,0.054026
58,HPU,0.005290,0.007805,0.005250,0.006734,0.008257,0.005834,0.052869
